# Notebook 10: Dimensionality Reduction

## Why This Matters
Embeddings live in hundreds of dimensions. That's hard to inspect, expensive to index, and slow to search. Dimensionality reduction serves two distinct purposes: **visualization** (understanding your embedding space) and **compression** (making search faster without losing retrieval quality). The right tool depends on which one you need.

## Structure
1. PCA vs UMAP vs t-SNE — what each optimizes (narrative)
2. PCA — linear projection, exact geometry
3. UMAP — topology-preserving nonlinear reduction
4. t-SNE — cluster visualization (narrative on pitfalls)
5. Compression: how many dimensions until retrieval degrades?
6. Bridge to ANN

## What You'll Learn
- What PCA, UMAP, and t-SNE each preserve and distort
- Why t-SNE cluster sizes and distances are meaningless
- How to pick a target dimension for compression without hurting retrieval
- When to use each method in production


## Background

### What each method optimizes

**PCA** finds the linear directions of maximum variance. Distances in PCA space are interpretable — the projection is orthogonal and preserves global structure. Fast (SVD on covariance matrix), deterministic, and reversible. The right choice for compression and for cases where you want to inspect what the principal components represent.

**UMAP** (McInnes et al., 2018) optimizes a fuzzy topological representation — it tries to preserve the high-dimensional neighborhood structure in the low-dimensional projection. Better than t-SNE for global structure, faster, and can be used for both visualization and compression. The default for embedding visualization today.

**t-SNE** optimizes pairwise distances in a neighborhood, using a Student-t kernel in low-dimensional space to allow distant clusters to separate. It produces beautiful cluster visualizations but has critical pitfalls:
- **Cluster sizes are meaningless** — all clusters are pulled to similar sizes
- **Distances between clusters are meaningless** — only within-cluster structure is preserved
- **Not deterministic** — results vary between runs
- **Not invertible** — can't compress and reconstruct
- **Slow** — O(N log N) at best, O(N²) naively

t-SNE is appropriate for a specific use case: understanding whether a dataset has cluster structure. It's often misused to make impressive-looking visualizations that don't actually tell you anything about the embedding space.


In [ ]:
# %pip install -q sentence-transformers umap-learn scikit-learn numpy matplotlib torch

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer, util
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
import umap

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# A semantically structured corpus with clear groups
sentences = {
    "inference": [
        "Flash attention reduces memory using tiling", "KV cache stores past attention tensors",
        "Speculative decoding uses a draft model", "Continuous batching fills idle GPU slots",
        "PagedAttention manages memory with virtual paging",
    ],
    "training": [
        "Gradient checkpointing trades compute for memory", "Mixed precision uses FP16 and FP32",
        "LoRA fine-tunes low-rank weight updates", "RLHF aligns models with human feedback",
        "DPO directly optimizes preference pairs",
    ],
    "architecture": [
        "Self-attention computes queries keys and values", "The residual stream accumulates representations",
        "LayerNorm stabilizes activations across features", "MLP blocks expand and contract hidden states",
        "RoPE encodes position via complex rotation",
    ],
    "evaluation": [
        "Perplexity measures language model probability", "BLEU scores n-gram overlap with references",
        "ROUGE measures recall of reference n-grams", "BERTScore uses contextual embeddings",
        "Calibration measures confidence accuracy alignment",
    ],
}

all_texts  = [t for group in sentences.values() for t in group]
all_labels = [g for g, texts in sentences.items() for _ in texts]
color_map  = {"inference":"royalblue","training":"darkorange","architecture":"seagreen","evaluation":"crimson"}
colors     = [color_map[l] for l in all_labels]

embeddings = model.encode(all_texts)
print(f"Embeddings: {embeddings.shape}  ({len(all_texts)} texts × {embeddings.shape[1]} dims)")

## 1. PCA — Linear Reduction

In [ ]:
# Explained variance curve — how many PCs do you need?
pca_full = PCA()
pca_full.fit(embeddings)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(cumvar, color='royalblue')
axes[0].axhline(0.90, color='r', linestyle='--', label='90%')
axes[0].axhline(0.95, color='orange', linestyle='--', label='95%')
axes[0].set_xlabel("Number of components")
axes[0].set_ylabel("Cumulative explained variance")
axes[0].set_title("PCA: How Many Components?")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

dims_90 = np.argmax(cumvar >= 0.90) + 1
dims_95 = np.argmax(cumvar >= 0.95) + 1
print(f"Dims for 90% variance: {dims_90}")
print(f"Dims for 95% variance: {dims_95}")
print(f"Original dims:         {embeddings.shape[1]}")
print(f"Compression ratio (90%): {embeddings.shape[1]/dims_90:.1f}×")

# 2D PCA for visualization
pca_2d = PCA(n_components=2)
pca_coords = pca_2d.fit_transform(embeddings)

from matplotlib.patches import Patch
for i, (text, color, label) in enumerate(zip(all_texts, colors, all_labels)):
    axes[1].scatter(*pca_coords[i], c=color, s=60, zorder=2)
    axes[1].annotate(text[:25], pca_coords[i], fontsize=6, ha='center', va='bottom',
                     xytext=(0,3), textcoords='offset points')

axes[1].legend(handles=[Patch(color=c,label=l) for l,c in color_map.items()], loc='upper right', fontsize=8)
axes[1].set_title(f"PCA 2D ({sum(pca_2d.explained_variance_ratio_):.0%} variance)")
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2. UMAP — Topology-Preserving Nonlinear Reduction

In [ ]:
# UMAP for visualization
umap_2d = umap.UMAP(n_components=2, n_neighbors=5, min_dist=0.1, metric='cosine', random_state=42)
umap_coords = umap_2d.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(9, 6))
for i, (text, color) in enumerate(zip(all_texts, colors)):
    ax.scatter(*umap_coords[i], c=color, s=70, zorder=2)
    ax.annotate(text[:30], umap_coords[i], fontsize=7, ha='center', va='bottom',
                xytext=(0,3), textcoords='offset points')

ax.legend(handles=[Patch(color=c,label=l) for l,c in color_map.items()], loc='upper right')
ax.set_title("UMAP 2D — neighborhood-preserving nonlinear projection")
ax.grid(True, alpha=0.3); ax.axis('off')
plt.tight_layout(); plt.show()

print("UMAP vs PCA:")
print("  PCA:  linear, preserves global variance structure, interpretable axes")
print("  UMAP: nonlinear, preserves local neighborhood, better cluster separation")
print("  For visualization: UMAP usually looks more structured")
print("  For compression: PCA is faster and more predictable")

## 3. t-SNE Pitfalls (Narrative)

t-SNE produces striking visualizations, but the most common mistake is over-interpreting them:

```python
# What t-SNE preserves:
# ✓ Whether clusters exist (structure within local neighborhoods)

# What t-SNE does NOT preserve:
# ✗ Cluster sizes (all clusters stretched to similar visual size)
# ✗ Distances between clusters (position is arbitrary)
# ✗ Global structure (far-apart points have no meaningful relationship)
# ✗ Reproducibility (random initialization → different layouts each run)
```

**The right interpretation:** if t-SNE shows clear clusters, your data probably has cluster structure. That's the only reliable conclusion. Don't report t-SNE cluster separations as evidence of semantic distance between groups.

**When to use t-SNE:**
- Exploring whether a dataset has categorical structure
- Debugging: do embeddings of the same class cluster together?
- Communication: showing cluster structure to non-technical stakeholders

**When not to use t-SNE:**
- Measuring how far apart two clusters are
- Compressing embeddings for search
- Any quantitative analysis of distances


## 4. Compression: How Many Dimensions Before Retrieval Degrades?

In [ ]:
# Measure recall@5 as a function of PCA dimensionality
def recall_at_k(embeddings, labels, k=5):
    sims = cosine_similarity(embeddings)
    np.fill_diagonal(sims, -1)
    n_correct = 0
    for i in range(len(embeddings)):
        top_k = np.argsort(sims[i])[::-1][:k]
        if any(labels[j] == labels[i] for j in top_k):
            n_correct += 1
    return n_correct / len(embeddings)

baseline_recall = recall_at_k(embeddings, all_labels, k=5)
print(f"Baseline recall@5 (full {embeddings.shape[1]}d): {baseline_recall:.3f}")
print()
print(f"{'Dims':>6}  {'% of original':>14}  {'Recall@5':>10}  {'vs baseline':>12}")
print("-" * 50)

target_dims = [2, 4, 8, 16, 32, 64, 128, 192, 256, 320, embeddings.shape[1]]
for d in target_dims:
    if d >= embeddings.shape[1]:
        emb_d = embeddings
    else:
        pca_d = PCA(n_components=d)
        emb_d = pca_d.fit_transform(embeddings)
    r = recall_at_k(emb_d, all_labels, k=5)
    pct = d / embeddings.shape[1] * 100
    delta = r - baseline_recall
    print(f"{d:>6}  {pct:>13.0f}%  {r:>10.3f}  {delta:>+11.3f}")

print()
print("Rule of thumb: 50-75% of dims often retains >95% of retrieval quality.")
print("Check this on your actual data — it varies significantly by corpus.")

## 5. When to Use Each

| Method | Use for | Don't use for |
|--------|---------|---------------|
| **PCA** | Compression, inspection of principal directions | When linear projection misses nonlinear structure |
| **UMAP** | Visualization with global structure preserved | Compression (slow to transform new points) |
| **t-SNE** | Checking for cluster structure | Distance analysis, compression, reproducibility |

**Production compression:** PCA with 50–75% of original dims, verified with domain recall@k.

**Visualization:** UMAP with `metric='cosine'`, `n_neighbors` tuned to your corpus size.


## 6. Bridge to Approximate Nearest Neighbors

After dimensionality reduction, you have compact embeddings ready for search. Exact nearest neighbor search is O(N·D) per query — fine for thousands of documents, prohibitive for millions.

Approximate Nearest Neighbor (ANN) algorithms trade a small amount of recall for massive speed gains. Notebook 11 covers FAISS — Facebook's production ANN library — including flat (exact) search, IVF (inverted file), HNSW, and product quantization. Together these let you search billions of vectors in milliseconds.


## Summary

| Method | Preserves | Distorts | Speed | Use |
|--------|-----------|----------|-------|-----|
| PCA | Global variance, linear distances | Nonlinear cluster structure | Fast | Compression, inspection |
| UMAP | Local neighborhoods, global topology | Absolute distances | Moderate | Visualization |
| t-SNE | Local cluster membership | Cluster sizes, inter-cluster distances | Slow | Cluster existence check only |

**Next:** Notebook 11 — Approximate Nearest Neighbors. FAISS flat → IVF → HNSW: how to search millions of vectors in milliseconds with controllable recall–speed tradeoffs.
